# Weather Data Analysis
**Problem statement:** Which weather conditions reduce visibility, how do wind, humidity and
temperature change across the year, and what do the unusual events (fog, snow) look like?

**Dataset columns:** `Date/Time`, `Temp_C`, `Dew Point Temp_C`, `Rel Hum_%`,
`Wind Speed_km/h`, `Visibility_km`, `Press_kPa`, `Weather`

**Questions**
1. Which weather conditions have the lowest visibility?
2. Which conditions come with the strongest wind?
3. How do temperature and humidity change by month and season?
4. When does snow or fog occur?
5. How often is it clear *and* humid, or very high visibility? (your original filters, corrected)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_PATH = "filewether.csv"
os.makedirs("images", exist_ok=True)

data = pd.read_csv(DATA_PATH)
data.head()

## 1. Inspect the data

In [ ]:
print("Shape:", data.shape)
print(data.dtypes)
data.info()

In [ ]:
data.describe()

## 2. Clean the data
Steps: rename `Weather`, convert dates, remove duplicates, handle nulls, tidy text, flag outliers.

In [ ]:
df = data.copy()

# Rename 
df = df.rename(columns={"Weather": "Weather_Condition"})

# Dates
df["Date/Time"] = pd.to_datetime(df["Date/Time"], errors="coerce")
print("Unparseable dates:", df["Date/Time"].isna().sum())
df = df.dropna(subset=["Date/Time"])

# Duplicates
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

# Text consistency
df["Weather_Condition"] = df["Weather_Condition"].astype(str).str.strip()

# Nulls
print("Nulls per column:")
print(df.isnull().sum())
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].interpolate(limit_direction="both") 
df = df.sort_values("Date/Time").reset_index(drop=True)

#Helper columns
df["Month"] = df["Date/Time"].dt.month
df["Hour"] = df["Date/Time"].dt.hour
season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
df["Season"] = df["Month"].map(season_map)

In [ ]:
# Outlier check with the IQR rule (reported, not removed: weather extremes are real data)
for col in ["Temp_C", "Rel Hum_%", "Wind Speed_km/h", "Visibility_km", "Press_kPa"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col:18s} outliers: {n_out}")

print("Rows after cleaning:", len(df))

## 3. Exploratory analysis

In [ ]:
# Summary statistics (mean, std, variance )
print("Mean visibility:", round(df["Visibility_km"].mean(), 2))
print("Std of pressure:", round(df["Press_kPa"].std(), 3))
print("Variance of humidity:", round(df["Rel Hum_%"].var(), 2))
print("Unique wind speeds:", df["Wind Speed_km/h"].nunique())

In [ ]:
# Chart 1: how often each weather condition occurs
counts = df["Weather_Condition"].value_counts().head(10)
ax = counts.sort_values().plot(kind="barh", figsize=(8, 5), color="steelblue")
ax.set_title("Top 10 weather conditions")
ax.set_xlabel("Number of hourly records")
plt.tight_layout()
plt.savefig("images/weather_counts.png", dpi=120)
plt.show()

In [ ]:
# Chart 2: distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ["Temp_C", "Dew Point Temp_C", "Rel Hum_%",
                                  "Wind Speed_km/h", "Visibility_km", "Press_kPa"]):
    ax.hist(df[col], bins=30, color="teal", edgecolor="white")
    ax.set_title(col)
plt.tight_layout()
plt.savefig("images/distributions.png", dpi=120)
plt.show()

In [ ]:
# Chart 3: temperature trend over time
monthly = df.set_index("Date/Time")["Temp_C"].resample("MS").mean()
ax = monthly.plot(figsize=(10, 4), marker="o", color="firebrick")
ax.set_title("Average temperature by month")
ax.set_ylabel("Temp (C)")
plt.tight_layout()
plt.savefig("images/temp_trend.png", dpi=120)
plt.show()

In [ ]:
# Chart 4: correlation heatmap
corr = df[["Temp_C", "Dew Point Temp_C", "Rel Hum_%", "Wind Speed_km/h",
           "Visibility_km", "Press_kPa"]].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im)
ax.set_title("Correlation between measurements")
plt.tight_layout()
plt.savefig("images/correlation.png", dpi=120)
plt.show()

## 4. Answering the questions

In [ ]:
# Q1: which conditions have the lowest visibility? (min 20 records to avoid noise)
by_weather = df.groupby("Weather_Condition").agg(
    records=("Visibility_km", "size"),
    mean_visibility=("Visibility_km", "mean"),
    mean_wind=("Wind Speed_km/h", "mean"),
    mean_temp=("Temp_C", "mean"),
    mean_humidity=("Rel Hum_%", "mean"),
).round(2)
by_weather = by_weather[by_weather["records"] >= 20]

q1 = by_weather.sort_values("mean_visibility")
print(q1[["records", "mean_visibility"]].head(8))

ax = q1["mean_visibility"].head(10).sort_values(ascending=False).plot(
    kind="barh", figsize=(8, 5), color="darkorange")
ax.set_title("Average visibility by weather condition (lowest 10)")
ax.set_xlabel("Visibility (km)")
plt.tight_layout()
plt.savefig("images/visibility_by_weather.png", dpi=120)
plt.show()

In [ ]:
# Q2: which conditions come with the strongest wind?
q2 = by_weather.sort_values("mean_wind", ascending=False)
print(q2[["records", "mean_wind"]].head(8))

In [ ]:
# Q3: temperature and humidity by month and season
monthly_stats = df.groupby("Month")[["Temp_C", "Rel Hum_%", "Wind Speed_km/h"]].mean().round(2)
print(monthly_stats)
season_stats = df.groupby("Season")[["Temp_C", "Rel Hum_%", "Visibility_km"]].mean().round(2)
print(season_stats)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(monthly_stats.index, monthly_stats["Temp_C"], color="firebrick", marker="o", label="Temp (C)")
ax1.set_xlabel("Month")
ax1.set_ylabel("Temp (C)", color="firebrick")
ax2 = ax1.twinx()
ax2.plot(monthly_stats.index, monthly_stats["Rel Hum_%"], color="navy", marker="s", label="Humidity (%)")
ax2.set_ylabel("Humidity (%)", color="navy")
ax1.set_title("Temperature and humidity by month")
plt.tight_layout()
plt.savefig("images/month_temp_humidity.png", dpi=120)
plt.show()

In [ ]:
# Q4: when do snow and fog happen?
snow = df[df["Weather_Condition"].str.contains("Snow", case=False)]
fog = df[df["Weather_Condition"].str.contains("Fog", case=False)]
print("Snow records:", len(snow), "| mean temp:", round(snow["Temp_C"].mean(), 2))
print("Fog records:", len(fog), "| mean visibility:", round(fog["Visibility_km"].mean(), 2))
print("\nSnow by month:\n", snow["Month"].value_counts().sort_index())
print("\nFog by hour of day (top 5):\n", fog["Hour"].value_counts().head(5))

In [ ]:

clear_visible = df[(df["Weather_Condition"].str.lower() == "clear") & (df["Visibility_km"] > 40)]
print("Clear AND visibility > 40:", len(clear_visible))

clear_humid_or_far = df[
    ((df["Weather_Condition"].str.lower() == "clear") & (df["Rel Hum_%"] > 50))
    | (df["Visibility_km"] > 40)
]
print("(Clear AND humidity > 50) OR visibility > 40:", len(clear_humid_or_far))

windy_clear_sky = df[(df["Wind Speed_km/h"] > 24) & (df["Visibility_km"] == 25)]
print("Wind > 24 AND visibility = 25:", len(windy_clear_sky))

## 5. Key findings
Run the notebook on your data, then replace the placeholders below with the numbers you see.

1. **Visibility:** *(condition)* has the lowest average visibility at *(X)* km, compared with *(Y)* km for clear weather.
2. **Wind:** *(condition)* is linked with the strongest wind, averaging *(X)* km/h.
3. **Seasons:** *(month)* is the coldest and *(month)* the warmest; humidity peaks in *(season)*.
4. **Snow and fog:** snow appears mostly in *(months)* at around *(X)* C; fog is most common around *(hour)*.
5. **Correlations:** temperature and dew point are strongly related; humidity and visibility *(positive/negative)* relationship of *(r)*.


